# Similarity threshold for article search

Checks `DEFAULT_MIN_SIMILARITY` in `src/db/queries.py`. Runs a few relevant
queries and one unrelated control query **without** a threshold and shows the
similarity of the top hits. A good threshold sits above the control query's
best scores and below the relevant queries' useful hits.

Needs Postgres running with embedded articles and `GCP_PROJECT_ID` in `.env`.

In [1]:
import sys
from datetime import date
from pathlib import Path

from dotenv import load_dotenv

sys.path.insert(0, str(Path.cwd().parent))  
load_dotenv(Path.cwd().parent / ".env")

from src.db.queries import DEFAULT_MIN_SIMILARITY
from src.db.session import get_session
from src.retriever.search import search_articles

In [2]:
TICKER = "AAPL"
AS_OF = date(2026, 1, 15)
QUERIES = [
    "quarterly earnings results and guidance",
    "lawsuit, fine or regulatory investigation",
    "new product launch or partnership",
    "analyst upgrade or downgrade, price target change",
]
CONTROL = "chocolate cake recipe with strawberries"  

In [3]:
def run(variant: str) -> dict[str, list[tuple[str, float]]]:
    results = {}
    with get_session() as session:
        for query in [*QUERIES, CONTROL]:
            rows = search_articles(
                session, TICKER, AS_OF, query,
                limit=10, variant=variant, min_similarity=None,
            )
            results[query] = [(a.title or "", sim) for a, sim in rows]
    return results


def report(variant: str) -> None:
    results = run(variant)
    print(f"===== {TICKER}, as of {AS_OF}, variant: {variant}")
    for query, rows in results.items():
        print(f"\n{query}")
        if not rows:
            print("  (no articles)")
            continue
        scores = [sim for _, sim in rows]
        print(f"  top {max(scores):.3f}  10th {min(scores):.3f}")
        for title, sim in rows[:5]:
            print(f"  {sim:.3f}  {title[:90]}")

    control_top = max((s for _, s in results[CONTROL]), default=0.0)
    relevant_low = min(
        (min(s for _, s in rows) for q, rows in results.items() if q != CONTROL and rows),
        default=1.0,
    )
    print(
        f"\ncontrol best {control_top:.3f} | relevant 10th-hit worst {relevant_low:.3f}"
        f" | current threshold {DEFAULT_MIN_SIMILARITY}"
        f" -> {'OK' if control_top < DEFAULT_MIN_SIMILARITY <= relevant_low else 'CHECK'}"
    )

In [4]:
report("title_summary")

===== AAPL, as of 2026-01-15, variant: title_summary

quarterly earnings results and guidance
  top 0.653  10th 0.596
  0.653  What You Need To Know Ahead Of Apple's Earnings Release - Barchart.com
  0.643  What You Need To Know Ahead of Apple's Earnings Release - inkl
  0.637  Zacks Research Issues Optimistic Outlook for Apple Earnings - Defense World
  0.620  Apple and Amazon Earnings Beat Expectations on Cloud and Services Strength
  0.618  Will Apple (AAPL) Beat Estimates Again in Its Next Earnings Report?

lawsuit, fine or regulatory investigation
  top 0.580  10th 0.528
  0.580  Apple hit with EPA penalty over claimed hazardous waste management violations
  0.580  Apple settles with EPA regarding hazardous waste management
  0.555  EU Commission Prepares Apple Investigation After Complaint
  0.552  Apple fined €98.6m by Italian Competition Authority
  0.541  ETtech Explainer: Why Apple is fighting India’s heavy-fine rule

new product launch or partnership
  top 0.558  10th 0.513


In [5]:
report("summary")

===== AAPL, as of 2026-01-15, variant: summary

quarterly earnings results and guidance
  top 0.641  10th 0.593
  0.641  What You Need To Know Ahead Of Apple's Earnings Release - Barchart.com
  0.633  What You Need To Know Ahead of Apple's Earnings Release - inkl
  0.631  Zacks Research Issues Optimistic Outlook for Apple Earnings - Defense World
  0.620  Will Apple (AAPL) Beat Estimates Again in Its Next Earnings Report?
  0.618  Investors Heavily Search Apple Inc. (AAPL): Here is What You Need to Know

lawsuit, fine or regulatory investigation
  top 0.569  10th 0.522
  0.569  Apple settles with EPA regarding hazardous waste management
  0.566  Apple hit with EPA penalty over claimed hazardous waste management violations
  0.543  Apple fined €98.6m by Italian Competition Authority
  0.541  ETtech Explainer: Why Apple is fighting India’s heavy-fine rule
  0.537  EU Commission Prepares Apple Investigation After Complaint

new product launch or partnership
  top 0.539  10th 0.509
  0.539